In [93]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../config"
import lseg.data as ld
import pandas as pd

In [89]:
START = "2000-01-01T06:00:00"
END = "2025-10-31T19:30:00"
trading_fields = ["TR.OPENPRICE", "TR.CLOSEPRICE"]
daily_fields = ["TR.WACCBeta"]
sector_fields = [
    "TR.TRBCEconomicSectorAll", 
    "TR.TRBCBusinessSectorAll", 
    "TR.TRBCIndustryGroupAll",
    "TR.TRBCIndustryAll",
    "TR.TRBCActivityAll"]

In [73]:
ld.open_session()

<lseg.data.session.Definition object at 0x1ef886377a0 {name='workspace'}>

In [97]:
pd.set_option('future.no_silent_downcasting', True)
a = ld.get_history(
    universe='ELSE.OQ',
    fields=trading_fields,
    interval="daily",
    start=START,
    end=END
    )

In [96]:
a

DJT.OQ,Open Price,Close Price
Date,,
2000-01-03,<NA>,<NA>
2000-01-04,<NA>,<NA>
2000-01-05,<NA>,<NA>
2000-01-06,<NA>,<NA>
2000-01-07,<NA>,<NA>
...,...,...
2025-10-27,16.0,16.03
2025-10-28,16.51,16.16
2025-10-29,16.11,15.98


In [71]:
ld.get_history(
    universe='AAPL.O',
    fields=["TR.OPENPRICE"],
    start=START,
    end=END
    )

AAPL.O,Open Price
Date,
2000-01-03,0.936383
2000-01-04,0.966517
2000-01-05,0.926338
2000-01-06,0.947544
2000-01-07,0.861606
...,...
2025-10-27,264.88
2025-10-28,268.985
2025-10-29,269.275


In [60]:
a["Exchange Code"].dropna()

Date
2015-12-12     
2015-12-13     
2015-12-14     
2015-12-15     
2015-12-16     
             ..
2024-11-27     
2024-11-28     
2024-11-29     
2024-11-30     
2024-12-01     
Name: Exchange Code, Length: 3238, dtype: string

In [38]:
# compute daily return series
returns_df = a.sort_index().copy()
returns_df["Intraday Return"] = (returns_df["Close Price"] - returns_df["Open Price"]) / returns_df["Open Price"] * 100
returns_df["Overnight Return"] = (returns_df["Open Price"] - returns_df["Close Price"].shift(1)) / returns_df["Close Price"].shift(1) * 100
returns_df = returns_df.dropna(subset=["Overnight Return"])
returns_df[["Open Price", "Close Price", "Intraday Return", "Overnight Return"]]


AAPL.O,Open Price,Close Price,Intraday Return,Overnight Return
Date,,,,
2004-10-04,0.699642,0.692678,-0.995366,1.318832
2004-10-05,0.688571,0.703035,2.100582,-0.592916
2004-10-06,0.705356,0.725714,2.886202,0.33014
2004-10-07,0.723928,0.707499,-2.269425,-0.246102
2004-10-08,0.706249,0.697499,-1.23894,-0.176679
...,...,...,...,...
2024-11-22,228.06,229.87,0.793651,-0.201295
2024-11-25,231.46,232.87,0.609177,0.691695
2024-11-26,233.33,235.06,0.741439,0.197535


In [41]:
print(returns_df["Intraday Return"].mean())
print(returns_df["Intraday Return"].std())
print(returns_df["Overnight Return"].mean())
print(returns_df["Overnight Return"].std())


0.02627725864640391
1.6575451177846015
0.11069689157734515
1.2716428103467894


In [34]:
import yfinance as yf
import pandas as pd
spy = yf.download("SPY", start = "2000-01-01", end = "2025-11-01", auto_adjust = True)

[*********************100%***********************]  1 of 1 completed


In [35]:
spy = spy.xs('SPY', axis=1, level='Ticker').copy()
spy = spy.rename(columns = {"Close": "close_price", "Open": "open_price"})
spy = spy[["open_price", "close_price"]]
spy["intra_day_return"] = (spy["close_price"] - spy["open_price"]) / spy["open_price"]
spy["overnight_return"] = (spy["open_price"] - spy["close_price"].shift(1)) / spy["close_price"].shift(1)
spy = spy.dropna(subset=["overnight_return"])

In [37]:
spy.to_csv("S:/campaignfinance/data/market/returns/ASPY.csv")

In [13]:
spy.xs('SPY', axis=1, level='Ticker').copy()

Price,Close,High,Low,Open,Volume
Date,,,,,
2000-01-03,91.887794,93.664739,90.900603,93.664739,8164300
2000-01-04,88.294434,91.019083,88.225330,90.683438,8089800
2000-01-05,88.452362,89.419810,86.714905,88.412874,12177900
2000-01-06,87.030746,89.400005,87.030746,88.215375,6227200
2000-01-07,92.085220,92.085220,88.491843,88.649794,8066500
...,...,...,...,...,...
2025-10-24,677.250000,678.469971,675.650024,676.460022,74356500
2025-10-27,685.239990,685.539978,682.119995,682.729980,63339800
2025-10-28,687.059998,688.909973,684.830017,687.049988,61738100
